# Epoch-wise adaptive k-MIP: PascalVOC-SP and Peptides-func

Runs the paper's **fixed k=15** baseline and **linear 50 to 10** over 4 seeds, using paper-derived per-dataset training settings.

- **k changes over epochs** (paper Limitations suggestion: large → small).
- **Within an epoch, every layer uses the same k.**

**Free-tier tips**
- Runtime → Change runtime type → **T4 GPU**
- Mount Drive so disconnects don't wipe checkpoints
- Code is **cloned from GitHub** (this fork), not uploaded as a zip
- Research starts in full mode with the selected paper epoch horizon; quick/standard are independent runs.
- LapPE/RWSE stay **off**


## 0) Runtime check


In [1]:
import torch, psutil, os
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))
print('ram_gb:', round(psutil.virtual_memory().total/1e9, 2))
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → T4'


cuda: True
gpu: Tesla T4
vram_gb: 15.64
ram_gb: 13.61


## 1) Mount Drive (recommended)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/kmip_adaptive_k'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)


Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/kmip_adaptive_k


## 2) Clone the repo

Clones **fix/voc-superpixels-torch-load** from **https://github.com/ZoyaAnwar/adaptive-k-MIP-GraphGPS.git**: the paper linear-k baseline plus the PascalVOC-SP loader fixes (Hugging Face download, PyTorch ≥ 2.6 loading).  
Do **not** clone the upstream paper repo — it lacks this curriculum.

In [3]:
import os, shutil

# Leave /content/kmip before deleting it; otherwise re-running this cell fails
# with "Unable to read current working directory".
%cd /content
REPO = '/content/kmip'
if os.path.exists(REPO):
    shutil.rmtree(REPO)

# Clone the fix/voc-superpixels-torch-load branch explicitly.
!git clone --depth 1 --branch fix/voc-superpixels-torch-load https://github.com/ZoyaAnwar/adaptive-k-MIP-GraphGPS.git /content/kmip
%cd /content/kmip

assert os.path.exists('main.py'), 'clone failed — main.py missing'
assert os.path.exists('graphgps/adaptive_k/train_schedule.py'), 'epoch-k module missing'
assert os.path.exists('colab/setup_env.sh'), 'colab scripts missing'
print('Repo ready at', os.getcwd())
!git rev-parse --short HEAD


/content
Cloning into '/content/kmip'...
remote: Enumerating objects: 368, done.
remote: Counting objects: 100% (368/368), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 368 (delta 166), reused 340 (delta 157), pack-reused 0 (from 0)
Receiving objects: 100% (368/368), 767.80 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (166/166), done.
/content/kmip
Repo ready at /content/kmip
61105e5


### Optional: refresh after you push

Re-run the clone cell above, or:


In [ ]:
# !git -C /content/kmip pull
# %cd /content/kmip


## 3) Point datasets + results at Drive


In [4]:
%cd /content/kmip

import os
DRIVE_ROOT = '/content/drive/MyDrive/kmip_adaptive_k'
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

if os.path.islink('datasets') or os.path.exists('datasets'):
    if os.path.islink('datasets'):
        os.unlink('datasets')
    elif os.path.isdir('datasets') and not os.listdir('datasets'):
        os.rmdir('datasets')
if not os.path.exists('datasets'):
    os.symlink(f'{DRIVE_ROOT}/datasets', 'datasets')

print('datasets ->', os.path.realpath('datasets'))
print('cwd', os.getcwd())


/content/kmip
datasets -> /content/drive/MyDrive/kmip_adaptive_k/datasets
cwd /content/kmip


## 4) Install environment


In [5]:
!bash colab/setup_env.sh


== GPU / CUDA ==
Mon Oct 12 18:25:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

## 4b) Optional PascalVOC-SP download and check

The training run downloads PascalVOC-SP automatically from the LRGB Hugging Face mirror (~1.1 GB, once, into Drive). This cell does that ahead of time and checks the data against the paper (Table 4) and the encoders: x with 14 columns, edge_attr with 2, 21 classes, 11,355 graphs, ~479.4 nodes and ~2,710.5 edges per graph.

In [ ]:
RUN_VOC_CHECK = False  # Set True when running PascalVOC-SP.
if RUN_VOC_CHECK:
    from graphgps.loader.dataset.voc_superpixels import VOCSuperpixels

    tot_g = tot_n = tot_e = 0
    labels = set()
    for split in ['train', 'val', 'test']:
        ds = VOCSuperpixels(root='datasets/VOCSuperpixels', name='edge_wt_region_boundary',
                            slic_compactness=30, split=split)
        d = ds._data
        tot_g += len(ds); tot_n += d.x.shape[0]; tot_e += d.edge_index.shape[1]
        labels |= set(d.y.unique().tolist())
        print(f'{split}: graphs={len(ds)}  x={tuple(d.x.shape)} {d.x.dtype}  '
              f'edge_attr={tuple(d.edge_attr.shape)} {d.edge_attr.dtype}  y={d.y.dtype}')

    print(f'\ntotal graphs {tot_g}   (paper: 11,355)')
    print(f'avg nodes    {tot_n/tot_g:.1f}  (paper: 479.4)')
    print(f'avg edges    {tot_e/tot_g:.1f}  (paper: 2,710.5)')
    print(f'classes      {len(labels)} -> {sorted(labels)}  (paper: 21)')

## 5) Optional Peptides-func environment smoke test (1 epoch, tiny)

This existing test is independent of research and writes to results/smoke. Never resume it into full research training.


In [ ]:
RUN_PEPTIDES_SMOKE = False  # Manual opt-in: leave False for Run all and PascalVOC-SP runs.
if RUN_PEPTIDES_SMOKE:
    !python main.py \
      --cfg configs/LRGB/Peptides-func/colab/GPS+kmip-fixed.yaml \
      wandb.use False \
      optim.max_epoch 1 \
      train.batch_size 32 \
      model.auto_satisfy_param_budget False \
      gt.layers 2 gnn.layers_mp 2 \
      gt.dim_hidden 64 gnn.dim_inner 64 \
      gt.sparse.val_dim 16 gt.sparse.kq_dim 8 \
      out_dir /content/drive/MyDrive/kmip_adaptive_k/results/smoke \
      name_tag smoke


## 6) Select research dataset and run fixed k=15 and linear 50 to 10

| MODE | Epochs | When |
|------|--------|------|
| `quick` | 20 | independent short/debug run; selected config batch/model settings |
| `standard` | 100 | independent exploratory; selected config batch/model settings |
| `full` | config | PascalVOC-SP: 200 epochs / batch 50; Peptides-func: 250 / batch 200 |

| RUN | Meaning |
|-----|---------|
| `fixed` | default: constant k=15 every epoch (paper baseline, same config) |
| `linear` | default: k 50 to 10 over the selected paper epoch horizon |
| `exponential` | existing Peptides-func exploratory config; settings unchanged |

Seeds default to 0,1,2,3, matching the paper's four seeds. Before the first full run, a `MODE=quick SEEDS=0` run checks the pipeline (it writes to a separate folder).

Resume each seed with the same DATASET, MODE, config and OUT_ROOT; never change max_epoch. SEEDS selects training launches; REPORT_SEEDS independently selects reporting. Outputs are dataset/mode/run-separated.


In [ ]:
import os
os.environ['DATASET'] = 'PascalVOC-SP'  # PascalVOC-SP | Peptides-func
os.environ['MODE'] = 'full'             # full research | quick short/debug | standard exploratory
os.environ['RUNS'] = 'linear'     # fixed baseline + linear; exponential: Peptides-func only
os.environ['OUT_ROOT'] = '/content/drive/MyDrive/kmip_adaptive_k/results'
os.environ['SEEDS'] = '0,1,2,3'         # Training processes to launch.
os.environ['REPORT_SEEDS'] = '0,1,2,3'  # Completed seeds to include in reporting.

!bash colab/run_ablation.sh


ROOT=/content/kmip
OUT_ROOT=/content/drive/MyDrive/kmip_adaptive_k/results  DATASET=PascalVOC-SP  MODE=full  RUNS=linear  SEEDS=0,1,2,3

======== RUN linear ========
cfg=configs/LRGB/PascalVOC/defaults/GPS+kmip-linear-50-to-10.yaml  out_dir=/content/drive/MyDrive/kmip_adaptive_k/results/PascalVOC-SP/full/linear
-------- seed=0 (repeat=1) --------
/content/kmip/graphgps/encoder/signnet_pos_encoder.py:183: SyntaxWarning: invalid escape sequence '\p'
  SignNetPE(v_1, ... , v_k) = \rho ( [\phi(v_i) + \rhi(−v_i)]^k_i=1 )
/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
[*] Run ID 0: seed=0, split_index=0
    Starting now: 2026-10-12 18:27:39.950834
[*] Loaded dataset 'edge_wt_region_boundary' fro

## 7) Authoritative per-seed and aggregate test results (F1 / AP)

SEEDS chooses which training processes to launch; REPORT_SEEDS independently chooses which completed seeds to report. Both default to 0,1,2,3. For example, train SEEDS=4,5 and report REPORT_SEEDS=0,1,2,3,4,5.

Select each reported seed's earliest best validation epoch and matching test result. Compute test mean and population SD only when every REPORT_SEEDS seed is complete. Other seed directories are ignored with a warning. This does not use PyG aggregation outputs.


In [ ]:
import json, math, pathlib, re, statistics, warnings
import pandas as pd
import yaml

root = pathlib.Path(os.environ['OUT_ROOT']) / os.environ['DATASET'] / os.environ['MODE']
metric = {'PascalVOC-SP': 'f1', 'Peptides-func': 'ap'}[os.environ['DATASET']]
report_seed_text = os.environ.get('REPORT_SEEDS', '0,1,2,3')
if not re.fullmatch(r'(0|[1-9][0-9]*)(,(0|[1-9][0-9]*))*', report_seed_text):
    raise ValueError(f'Invalid REPORT_SEEDS={report_seed_text!r}; use comma-separated nonnegative integers.')
report_seeds = [int(seed) for seed in report_seed_text.split(',')]
if len(set(report_seeds)) != len(report_seeds):
    raise ValueError(f'Duplicate seeds in REPORT_SEEDS={report_seed_text!r}')


def read_metric_history(path, metric):
    if not path.is_file():
        raise ValueError(f'Missing history: {path}')
    by_epoch = {}
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f'Incomplete/invalid JSON: {path}:{line_number}') from error
        if not isinstance(record, dict):
            raise ValueError(f'Invalid metric record: {path}:{line_number}')
        epoch = record.get('epoch')
        value = record.get(metric)
        if type(epoch) is not int or epoch < 0:
            raise ValueError(f'Invalid absolute epoch: {path}:{line_number}')
        if type(value) not in (int, float) or not math.isfinite(value):
            raise ValueError(f'Missing/invalid {metric}: {path}:{line_number}')
        if epoch in by_epoch and by_epoch[epoch] != record:
            raise ValueError(f'Conflicting duplicate epoch {epoch}: {path}')
        by_epoch[epoch] = record
    if not by_epoch:
        raise ValueError(f'Empty history: {path}')
    return by_epoch


def summarize_seed(seed_dir, metric):
    val = read_metric_history(seed_dir / 'val' / 'stats.json', metric)
    test = read_metric_history(seed_dir / 'test' / 'stats.json', metric)
    # Current research configs evaluate every epoch. Reject gaps/mismatched splits.
    epochs = sorted(val)
    if epochs != list(range(epochs[-1] + 1)) or set(val) != set(test):
        raise ValueError(f'Incomplete/mismatched validation/test epochs: {seed_dir}')
    best_epoch = max(epochs, key=lambda epoch: val[epoch][metric])
    result = {
        'best_epoch': best_epoch,
        f'val_{metric}': val[best_epoch][metric],
        f'test_{metric}': test[best_epoch][metric],
        'history_last_epoch': epochs[-1],
        'status': 'completion_unverified',
        'note': 'Best across saved epochs; full-run completion cannot be verified without config.yaml.',
    }
    config_path = seed_dir.parent / 'config.yaml'
    if config_path.is_file():
        config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
        expected_epochs = config['optim']['max_epoch']
        if type(expected_epochs) is not int or expected_epochs < 1:
            raise ValueError(f'Invalid max_epoch: {config_path}')
        if epochs[-1] >= expected_epochs:
            raise ValueError(f'History exceeds saved epoch horizon: {seed_dir}')
        complete = len(epochs) == expected_epochs
        result.update(
            expected_epochs=expected_epochs,
            status='complete' if complete else 'incomplete',
            note='' if complete else 'Partial run: best across saved epochs only; resume before final comparison.',
        )
    return result


rows = []
aggregates = []
for run in [name.strip() for name in os.environ.get('RUNS', 'fixed,linear').split(',')]:
    run_root = root / run
    seed_dirs = sorted(path for path in run_root.glob('*/*')
                       if path.is_dir() and path.name.isdigit())
    by_seed = {}
    for path in seed_dirs:
        by_seed.setdefault(int(path.name), []).append(path)
    ignored_seeds = sorted(set(by_seed) - set(report_seeds))
    if ignored_seeds:
        warnings.warn(f'{run}: ignoring seeds outside REPORT_SEEDS: {ignored_seeds}')
    run_rows = []
    for seed in report_seeds:
        result = {'run': run, 'seed': seed}
        paths = by_seed.get(seed, [])
        if len(paths) != 1:
            result.update(
                status='error',
                error=f'Expected one directory for seed {seed}; found {len(paths)} under {run_root}',
            )
        else:
            seed_dir = paths[0]
            result['seed_dir'] = str(seed_dir)
            try:
                result.update(summarize_seed(seed_dir, metric))
            except (ValueError, KeyError, TypeError, OSError, yaml.YAMLError) as error:
                result.update(status='error', error=str(error))
        run_rows.append(result)
    rows.extend(run_rows)

    aggregate = {
        'run': run, 'metric': metric, 'report_seeds': report_seeds,
        'ignored_seeds': ignored_seeds,
        'n_complete': sum(row['status'] == 'complete' for row in run_rows),
        'std_ddof': 0, 'status': 'not_ready',
    }
    configuration_dirs = {str(path.parent) for seed in report_seeds
                          for path in by_seed.get(seed, [])}
    if len(configuration_dirs) > 1:
        aggregate['reason'] = 'Selected seeds span multiple configurations; no aggregate computed.'
    elif all(row['status'] == 'complete' for row in run_rows):
        values = [row[f'test_{metric}'] for row in run_rows]
        aggregate.update(
            status='complete', n_seeds=len(values),
            test_mean=statistics.mean(values),
            test_std=statistics.pstdev(values),  # Population standard deviation: ddof=0.
        )
    else:
        aggregate['reason'] = 'Every REPORT_SEEDS seed must have complete, validated histories.'
    aggregates.append(aggregate)

display(pd.DataFrame(rows))
display(pd.DataFrame(aggregates))
summary = {
    'dataset': os.environ['DATASET'], 'mode': os.environ['MODE'],
    'metric': metric, 'report_seeds': report_seeds,
    'per_seed': rows, 'aggregates': aggregates,
}
root.mkdir(parents=True, exist_ok=True)
out = root / 'summary.json'
out.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
print('wrote', out)


## Research resume and independent exploratory runs

1. Setup, optional independent tiny smoke test, then select DATASET and start `MODE=full RUNS=fixed,linear`
2. Later sessions: same DATASET, MODE, config and OUT_ROOT (auto-resume per seed); choose SEEDS to launch and REPORT_SEEDS to summarize independently
3. Keep the selected paper max_epoch unchanged; never promote quick/standard runs into full
4. RUNS defaults to fixed,linear; the existing Peptides-func exponential config remains available

Watch the log for lines like:
`epoch=0/249 progress=0.000 requested_k=50 on 8 layer(s) (schedule=linear)` for Peptides-func; PascalVOC-SP uses denominator 199
